# Phase 5: Automated Hyperparameter Tuning
This notebook applies `GridSearchCV` to optimize our top-performing tree-based algorithms, systemically evaluating combinations to extract maximum $R^2$ performance.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

### 1. Load splits and isolate features

In [3]:
train_df = pd.read_csv("../artifacts/train.csv")
test_df = pd.read_csv("../artifacts/test.csv")

for df in [train_df, test_df]:
    df['Pavement_Age'] = 2026 - df['Beläggningsår']

features = ['Spårdjup max 15', 'ÅDT fordon', 'Pavement_Age', 'Hastighetsgräns', 'Spårdjup max 17', 'Vägbredd']
X_train = train_df[features]
y_train = train_df['IRI höger']
X_test = test_df[features]
y_actual = test_df['IRI höger']

### 2. Define Hyperparameter Tuning Search Grids

In [4]:
tuning_configs = {
    "Random_Forest_Tuned": {
        "model": RandomForestRegressor(random_state=42, n_jobs=-1),
        "params": {
            "n_estimators": [50, 100, 150],
            "max_depth": [6, 10, None],
            "min_samples_split": [2, 5]
        }
    },
    "Gradient_Boosting_Tuned": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            "n_estimators": [100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [4, 6]
        }
    }
}

### 3. Run Grid Search Optimization Loop

In [5]:
tuning_results = []
best_overall_r2 = -1
best_overall_model = None
best_overall_name = ""

for name, config in tuning_configs.items():
    print(f"Optimizing hyperparameters for {name}...")
    
    grid_search = GridSearchCV(
        estimator=config["model"], 
        param_grid=config["params"], 
        cv=3, 
        scoring='r2', 
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    optimized_model = grid_search.best_estimator_
    y_pred = optimized_model.predict(X_test)
    
    tuned_r2 = r2_score(y_actual, y_pred)
    tuned_mae = mean_absolute_error(y_actual, y_pred)
    tuned_rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    
    # Append stats to local results summary
    tuning_results.append({
        "Model Name": name,
        "Best Params": grid_search.best_params_,
        "Test R² Score": f"{tuned_r2*100:.2f}%",
        "MAE": round(tuned_mae, 4),
        "RMSE": round(tuned_rmse, 4)
    })
    
    # Cache checkpoint
    with open(f"../artifacts/model_{name}.pkl", "wb") as f:
        pickle.dump(optimized_model, f)
        
    if tuned_r2 > best_overall_r2:
        best_overall_r2 = tuned_r2
        best_overall_model = optimized_model
        best_overall_name = name

Optimizing hyperparameters for Random_Forest_Tuned...
Optimizing hyperparameters for Gradient_Boosting_Tuned...


### 4. Display Performance Summary Matrix

In [6]:
summary_df = pd.DataFrame(tuning_results)
summary_df.head()

,Model Name,Best Params,Test R² Score,MAE,RMSE
0,Random_Forest_Tuned,"{'max_depth': None, 'min_samples_split': 5, 'n...",50.73%,0.6911,1.1034
1,Gradient_Boosting_Tuned,"{'learning_rate': 0.1, 'max_depth': 6, 'n_esti...",49.40%,0.7050,1.1182


### 5. Serialize Production Winner

In [ ]:
print(f"🏆 Overall Tuning Champion: {best_overall_name} with {best_overall_r2*100:.2f}%")

output_model_path = "../artifacts/model.pkl"
with open(output_model_path, "wb") as f:
    pickle.dump(best_overall_model, f)
print(f"Production baseline saved to: {output_model_path}")